In [ ]:
import random, json, pandas as pd, os, re, itertools, math, hashlib
random.seed(42)

categories = [
    ("정치", ["정책", "국회", "청와대", "정치일반"]),
    ("경제", ["경제일반", "금융", "부동산", "산업"]),
    ("사회", ["사회일반", "사건사고", "교육", "노동"]),
    ("생활&문화", ["생활일반", "건강", "여행", "문화"]),
    ("IT&과학", ["IT일반", "모바일", "과학", "보안"]),
    ("세계", ["국제일반", "아시아", "미주", "유럽"]),
    ("연예", ["연예일반", "방송", "가요", "영화"]),
]

N_TOTAL = 500
base = N_TOTAL // len(categories)   # 71
rem = N_TOTAL % len(categories)     # 3

# Title building blocks (clickbait-y but not explicit falsehood in body)
subjects = [
    "청년", "20대", "직장인", "자영업자", "신혼부부", "대학생", "취준생", "1인가구", "고령층",
    "전세 세입자", "월세 거주자", "소상공인", "구직자", "근로자", "프리랜서", "육아 가정"
]
hooks = [
    "최대", "조건 따라", "대상 확대", "기준 변경", "전격 검토", "논의 급물살", "가능성", "방안",
    "혜택", "지원", "환급", "감면", "완화", "강화", "확대", "축소"
]
benefits = [
    "지원금", "환급", "감면", "바우처", "추가 혜택", "긴급 지원", "특별 지원", "부담 완화",
    "대출 우대", "금리 혜택", "세제 혜택", "수당", "포인트", "쿠폰"
]
numbers = [str(x) for x in range(10, 991, 5)]  # used only in titles

title_templates = [
    "{subj}라면 {num}만원 {benefit} 가능?",
    "{subj} 대상 {benefit} {num}만원까지…{hook} 나왔다",
    "{subj} {benefit}, {hook}되나…'조건'에 관심",
    "{subj} {benefit} {hook}…최대 {num}만원 거론",
    "{subj}에 {benefit} {hook}…'기준' 달라지나",
    "{subj} {hook}…{benefit} '최대 {num}만원' 언급",
    "이번엔 {subj}? {benefit} {hook}…{num}만원까지 가능성",
]

# One-sentence "empty" content templates (no numbers, no concrete steps)
content_templates = [
    "관계 부처는 관련 사안을 두고 여러 의견을 종합해 검토 중인 것으로 전해졌다.",
    "정부는 다양한 가능성을 열어두고 향후 논의를 이어갈 방침인 것으로 알려졌다.",
    "전문가들은 방향성 자체에 주목하면서도 구체 내용은 더 지켜봐야 한다고 했다.",
    "관련 업계는 기대와 우려가 엇갈린다며 향후 발표에 관심이 모인다고 전했다.",
    "당국은 세부 기준과 일정은 아직 확정되지 않았다고 설명한 것으로 전해졌다.",
    "논의가 진행 중인 만큼 현 단계에서는 단정하기 어렵다는 의견이 나온다.",
    "관계자들은 검토 단계라는 점을 강조하며 추가 논의 가능성을 시사했다.",
    "일각에서는 변화 가능성이 거론되지만 현재로서는 확정된 바가 없다는 반응이다.",
    "당사자들의 관심이 커지는 가운데 관련 논의가 이어지고 있다.",
    "관련 내용은 내부 검토 절차를 거쳐 정리될 것으로 보인다는 관측이 나온다.",
]

# Slight category flavor words that still don't add real info
flavors = {
    "정치": ["정책 방향", "논의 과정", "입장 조율", "후속 검토"],
    "경제": ["시장 반응", "전망", "논의 흐름", "관심"],
    "사회": ["현장 반응", "여론", "논의 경과", "관심"],
    "생활&문화": ["관심", "반응", "논의", "검토"],
    "IT&과학": ["업계 반응", "기술 논의", "검토", "전망"],
    "세계": ["국제적 관심", "논의", "반응", "관측"],
    "연예": ["관심", "반응", "논의", "관측"],
}

def make_content(cat):
    base = random.choice(content_templates)
    # inject a harmless flavor phrase by replacing one generic part, keeping one sentence.
    # Ensure we do NOT add numbers or concrete instructions.
    fl = random.choice(flavors[cat])
    # Insert flavor near the middle if possible
    if "관련" in base:
        return base.replace("관련", f"{fl} 관련", 1)
    else:
        return f"{fl}과 관련해 {base}"

def make_title():
    tpl = random.choice(title_templates)
    return tpl.format(
        subj=random.choice(subjects),
        num=random.choice(numbers),
        benefit=random.choice(benefits),
        hook=random.choice(hooks),
    ).replace("??", "?")

# Generate balanced counts
rows = []
seen = set()

def normalize(s):
    return re.sub(r"\s+", " ", s.strip())

for i, (cat, subcats) in enumerate(categories):
    k = base + (1 if i < rem else 0)  # first 3 get 72
    for _ in range(k):
        # try until unique
        for attempt in range(200):
            sub = random.choice(subcats)
            title = normalize(make_title())
            content = normalize(make_content(cat))
            key = (cat, sub, title, content, 0)
            if key in seen:
                continue
            # also ensure title+content pair unique globally
            key2 = (title, content)
            if key2 in {(r["newsTitle"], r["newsContent"]) for r in rows}:
                continue
            seen.add(key)
            rows.append({
                "newsCategory": cat,
                "newsSubcategory": sub,
                "newsTitle": title,
                "newsContent": content,
                "useType": 0
            })
            break
        else:
            raise RuntimeError("유니크 샘플 생성 실패")

len(rows), {c: sum(1 for r in rows if r["newsCategory"]==c) for c,_ in categories}



In [ ]:
# 저장: 06_evidence.ipynb가 읽는 파일명 · 5개 공통 필드
OUT_DIR = "../data/impact"
os.makedirs(OUT_DIR, exist_ok=True)
df = pd.DataFrame(rows, columns=["newsCategory", "newsSubcategory", "newsTitle", "newsContent", "useType"])
df.to_csv(os.path.join(OUT_DIR, "impact1_dataset_500.csv"), index=False, encoding="utf-8-sig")
df.to_json(os.path.join(OUT_DIR, "impact1_dataset_500.json"), orient="records", force_ascii=False, indent=2)
print("Saved:", len(df), "rows")
